In [39]:
import sqlite3
from difflib import SequenceMatcher

import pandas as pd
from datasets import load_from_disk

In [40]:
# 1. Dataset
DB_PATH = "../data/raw/banque_entreprise.db"
ds = load_from_disk("../data/processed/hf_dataset")

In [43]:
#2. Check des requête SQL du jeu de test

#Check de la bonne exécution du jeu de test ET avec des réponses non nulles
def run(sql):
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        return conn.execute(sql).fetchall()
    except Exception:
        return None
    finally:
        conn.close()


conn = sqlite3.connect(DB_PATH)
SCHEMA = "\n".join(r[0] for r in conn.execute(
    "SELECT sql FROM sqlite_master WHERE type='table'"))
conn.close()

toutes_les_requetes_sont_ok = True
print("### Test de fonctionnalité des requêtes du jeu de test : ###")

for ex in ds["test"]:
    res = run(ex["sql"])
    if res is None:
        print("ERREUR SQL :", ex["question"])
        toutes_les_requetes_sont_ok = False
    elif len(res) == 0:
        print("VIDE :", ex["question"])
        toutes_les_requetes_sont_ok = False


if toutes_les_requetes_sont_ok:
    print("Tout est OK : aucune erreur et aucun résultat vide !")
print("\n")


#Check des requêtes similaires entre les jeu de train/validation et test
ref = [ex["question"].lower() for s in ("train", "validation") for ex in ds[s]]
print("### Requêtes similaires entre train/validation et test : ###")

for ex in ds["test"]:
    q = ex["question"].lower()
    best = max(ref, key=lambda r: SequenceMatcher(None, q, r).ratio())
    score = SequenceMatcher(None, q, best).ratio()
    if score > 0.85:
        print(f"{score:.2f} | TEST : {ex['question']}\n       | TRAIN: {best}\n")
print("\n")

#Check des doublons entre les jeu de train/validation  et test      
sql_train = {ex["sql"].strip().lower() for s in ("train", "validation") for ex in ds[s]}
doublons = [ex["question"] for ex in ds["test"] if ex["sql"].strip().lower() in sql_train]
print("### Check des doublons entre train/validation et test : ###")
print(len(doublons), "SQL du test présents dans train/validation")

### Test de fonctionnalité des requêtes du jeu de test : ###
Tout est OK : aucune erreur et aucun résultat vide !


### Requêtes similaires entre train/validation et test : ###
0.88 | TEST : Quels clients ont une adresse email contenant gmail ?
       | TRAIN: quels clients ont une adresse email contenant le mot lopes ?

0.85 | TEST : Quelles factures ont été émises en juin 2026 ?
       | TRAIN: quelles factures ont été créées en 2025 ?

0.86 | TEST : Quels clients se sont inscrits un lundi ?
       | TRAIN: quels clients se sont inscrits en 2026 ?



### Check des doublons entre train/validation et test : ###
0 SQL du test présents dans train/validation
